# Enumerações (`enum`) e `typedef` — Tutorial

**Programação C (COMP0512) — UFS — 2026.2**

Este tutorial continua de onde a aula parou: as **tarefas de modificação** e o **desafio**, e
depois traz uma sequência de **exercícios com registros** que usam `struct`, `enum` e `typedef`
juntos.

## Objetivos

Ao final deste tutorial você será capaz de:

- declarar enumerações, com e sem valores explícitos, e prever o valor de cada constante;
- usar `enum` em `switch` e como índice de vetores de agregação;
- criar apelidos de tipo com `typedef` para `struct`, `enum` e tipos escalares;
- reconhecer os limites de cada um: `enum` não trava valores, `typedef` não cria tipo novo;
- modelar registros em que cada campo diz o que significa.

## Como usar

Cada célula de código escreve um arquivo `.c` (com `%%writefile`), compila com `gcc` e executa.
Antes de rodar qualquer célula marcada com **Preveja**, escreva a saída esperada — em papel ou
num comentário — e só depois execute.

In [ ]:
# Confira se o compilador está disponível no seu ambiente
!gcc --version | head -1

## 1. O programa da aula

Recompile e execute o programa condutor **exatamente como saiu da aula** e confirme que a saída
bate com a que vimos em sala.

In [ ]:
%%writefile ficha_situacao.c
#include <stdio.h>

typedef enum { REPROVADO, RECUPERACAO, APROVADO } Situacao;

typedef struct {
    char     nome[20];
    int      matricula;
    double   media;
    Situacao situacao;
} Aluno;

Situacao classifica(double media) {
    if (media >= 7.0) return APROVADO;
    if (media >= 5.0) return RECUPERACAO;
    return REPROVADO;
}

void imprime_situacao(Situacao s) {
    switch (s) {
        case APROVADO:    printf("Aprovado");    break;
        case RECUPERACAO: printf("Recuperacao"); break;
        case REPROVADO:   printf("Reprovado");   break;
    }
}

int main(void) {
    Aluno a = {"Ana Souza", 202600123, 8.75, REPROVADO};
    a.situacao = classifica(a.media);
    printf("%s (%d) media %.2f\n", a.nome, a.matricula, a.media);
    printf("situacao: %d = ", a.situacao);
    imprime_situacao(a.situacao);
    printf("\nsizeof(Situacao) = %zu\n", sizeof(Situacao));
    return 0;
}

In [ ]:
!gcc -Wall ficha_situacao.c -o ficha_situacao && ./ficha_situacao

In [ ]:
# Verificação automática da saída
saida = !./ficha_situacao
esperado = ["Ana Souza (202600123) media 8.75",
            "situacao: 2 = Aprovado",
            "sizeof(Situacao) = 4"]
print("OK" if saida == esperado else "Diferente do esperado:\n" + "\n".join(saida))

## 2. Investigando a enumeração

Três experimentos curtos. **Preveja a saída de cada um antes de executar.**

### 2.1 Que números o compilador atribuiu?

O programa abaixo imprime as constantes de três enumerações diferentes. Antes de rodar: quais
números aparecem em cada linha?

In [ ]:
%%writefile valores.c
#include <stdio.h>

typedef enum { REPROVADO, RECUPERACAO, APROVADO, NUM_SITUACOES } Situacao;
typedef enum { JAN = 1, FEV, MAR, ABR } Mes;
typedef enum { LEITURA = 1, ESCRITA = 2, EXECUCAO = 4 } Permissao;

int main(void) {
    printf("REPROVADO=%d RECUPERACAO=%d APROVADO=%d NUM_SITUACOES=%d\n",
           REPROVADO, RECUPERACAO, APROVADO, NUM_SITUACOES);
    printf("JAN=%d FEV=%d MAR=%d ABR=%d\n", JAN, FEV, MAR, ABR);
    printf("LEITURA|ESCRITA = %d\n", LEITURA | ESCRITA);
    return 0;
}

In [ ]:
!gcc -Wall valores.c -o valores && ./valores

**Responda no seu caderno:** por que `NUM_SITUACOES` vale exatamente a quantidade de situações
válidas? O que aconteceria com esse número se `REPROVADO` valesse 1?

### 2.2 O compilador cobra o `case` esquecido?

O `switch` abaixo trata só dois dos três valores. Compile com `-Wall` e leia a mensagem.

In [ ]:
%%writefile switch_incompleto.c
#include <stdio.h>

typedef enum { REPROVADO, RECUPERACAO, APROVADO } Situacao;

void imprime(Situacao s) {
    switch (s) {
        case APROVADO:  printf("Aprovado\n");  break;
        case REPROVADO: printf("Reprovado\n"); break;
    }
}

int main(void) { imprime(RECUPERACAO); return 0; }

In [ ]:
# O gcc compila mesmo assim: leia o AVISO antes de olhar a saida
!gcc -Wall switch_incompleto.c -o switch_incompleto
!./switch_incompleto
print("(nenhuma linha impressa acima = o switch nao tratou o valor recebido)")

### 2.3 `enum` trava os valores?

Preveja: o programa abaixo compila? Se compilar, o que imprime?

In [ ]:
%%writefile fora_da_faixa.c
#include <stdio.h>

typedef enum { REPROVADO, RECUPERACAO, APROVADO } Situacao;

int main(void) {
    Situacao s = 7;
    printf("s = %d\n", s);
    s = APROVADO + 1;
    printf("APROVADO + 1 = %d\n", s);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra fora_da_faixa.c -o fora_da_faixa && ./fora_da_faixa

**Conclusão a escrever com suas palavras:** o que uma enumeração garante e o que ela *não*
garante. Como você protegeria um campo `Situacao` preenchido a partir de um dado digitado
pelo usuário?

## 3. Altere o programa

As três tarefas da aula. Para cada uma: altere, **preveja**, recompile e compare.

### Tarefa 1 — acrescentar um valor à enumeração

Copie `ficha_situacao.c` para `tarefa1.c` e inclua `TRANCADO` em `Situacao`, antes da sentinela
(acrescente também a sentinela `NUM_SITUACOES`, se ainda não existir). **Não toque** em
`imprime_situacao` na primeira compilação: anote o que o `gcc -Wall` diz. Depois trate o novo caso.

In [ ]:
%%writefile tarefa1.c
#include <stdio.h>

/* TODO: acrescente TRANCADO e a sentinela NUM_SITUACOES */
typedef enum { REPROVADO, RECUPERACAO, APROVADO } Situacao;

typedef struct {
    char     nome[20];
    int      matricula;
    double   media;
    Situacao situacao;
} Aluno;

void imprime_situacao(Situacao s) {
    switch (s) {
        case APROVADO:    printf("Aprovado");    break;
        case RECUPERACAO: printf("Recuperacao"); break;
        case REPROVADO:   printf("Reprovado");   break;
        /* TODO: trate TRANCADO (so depois de ler o aviso do compilador) */
    }
}

int main(void) {
    Aluno a = {"Ana Souza", 202600123, 8.75, APROVADO};
    imprime_situacao(a.situacao);
    printf("\n");
    /* TODO: imprima quantos valores a enumeracao tem, usando a sentinela */
    return 0;
}

In [ ]:
!gcc -Wall tarefa1.c -o tarefa1
!./tarefa1

### Tarefa 2 — mexer nos valores

Em `tarefa2.c`, declare `REPROVADO = 1`. **Antes de rodar**, responda: o vetor
`cont[NUM_SITUACOES]` continua com tamanho suficiente? Qual índice fica sem uso? Alguma posição
fica fora do vetor?

Depois execute e confira sua previsão.

In [ ]:
%%writefile tarefa2.c
#include <stdio.h>

/* TODO: faca REPROVADO valer 1 */
typedef enum { REPROVADO, RECUPERACAO, APROVADO, NUM_SITUACOES } Situacao;

int main(void) {
    Situacao turma[6] = {APROVADO, REPROVADO, APROVADO,
                         RECUPERACAO, APROVADO, REPROVADO};
    int cont[NUM_SITUACOES] = {0};

    for (int i = 0; i < 6; i++)
        cont[turma[i]]++;

    printf("NUM_SITUACOES = %d\n", NUM_SITUACOES);
    for (int s = 0; s < NUM_SITUACOES; s++)
        printf("cont[%d] = %d\n", s, cont[s]);
    return 0;
}

In [ ]:
!gcc -Wall tarefa2.c -o tarefa2 && ./tarefa2

**Pergunta de fechamento:** qual regra você extrai daqui sobre usar `enum` como índice de vetor?
Se você *precisasse* começar a contagem em 1, como ajustaria o índice?

### Tarefa 3 — apelidar um tipo escalar

Em `tarefa3.c`, crie `typedef unsigned long Matricula` e troque `int matricula` por
`Matricula matricula`. Preveja `sizeof(Aluno)` **antes** de medir e explique a diferença usando o
que vimos sobre alinhamento na aula passada.

In [ ]:
%%writefile tarefa3.c
#include <stdio.h>
#include <stddef.h>

typedef enum { REPROVADO, RECUPERACAO, APROVADO } Situacao;

/* TODO: typedef unsigned long Matricula; */

typedef struct {
    char     nome[20];
    int      matricula;   /* TODO: troque para Matricula */
    double   media;
    Situacao situacao;
} Aluno;

int main(void) {
    printf("sizeof(Aluno) = %zu\n", sizeof(Aluno));
    printf("offset media    = %zu\n", offsetof(Aluno, media));
    printf("offset situacao = %zu\n", offsetof(Aluno, situacao));
    return 0;
}

In [ ]:
!gcc -Wall tarefa3.c -o tarefa3 && ./tarefa3

## 4. Exercícios com registros

Daqui em diante cada exercício parte de um esqueleto. Complete os `TODO`, compile com `-Wall`
(sem avisos!) e rode a célula de verificação que vem em seguida.

**Regra do exercício:** nenhum número mágico. Todo conjunto fechado de valores vira `enum`, e todo
registro ganha um apelido com `typedef`.

### Exercício 1 — Estoque por categoria

Modele um produto de supermercado e some o valor em estoque **por categoria**, usando um vetor
indexado pela enumeração.

Saída esperada:

```
Alimento: R$ 127.00
Limpeza: R$ 47.40
Bebida: R$ 96.00
Higiene: R$ 25.90
Total: R$ 296.30
```

In [ ]:
%%writefile ex1.c
#include <stdio.h>

typedef enum { ALIMENTO, LIMPEZA, BEBIDA, HIGIENE, NUM_CATEGORIAS } Categoria;

typedef struct {
    char      nome[30];
    Categoria categoria;
    double    preco;
    int       quantidade;
} Produto;

void imprime_categoria(Categoria c) {
    switch (c) {
        case ALIMENTO: printf("Alimento"); break;
        case LIMPEZA:  printf("Limpeza");  break;
        case BEBIDA:   printf("Bebida");   break;
        case HIGIENE:  printf("Higiene");  break;
        case NUM_CATEGORIAS: break;   /* sentinela: nunca e um valor real */
    }
}

int main(void) {
    Produto estoque[7] = {
        {"Arroz 5kg",     ALIMENTO, 27.50, 3},
        {"Feijao 1kg",    ALIMENTO,  8.90, 5},
        {"Detergente",    LIMPEZA,   2.90, 6},
        {"Sabao em po",   LIMPEZA,  15.00, 2},
        {"Refrigerante",  BEBIDA,    7.00, 8},
        {"Suco de uva",   BEBIDA,   10.00, 4},
        {"Sabonete",      HIGIENE,   2.59, 10}
    };
    double total_cat[NUM_CATEGORIAS] = {0.0};
    double total = 0.0;

    /* TODO 1: acumule preco * quantidade na posicao da categoria do produto */

    /* TODO 2: percorra a enumeracao de ALIMENTO ate NUM_CATEGORIAS imprimindo
       "<Categoria>: R$ <valor com 2 casas>" e somando em total */

    printf("Total: R$ %.2f\n", total);
    return 0;
}

In [ ]:
!gcc -Wall ex1.c -o ex1 && ./ex1

In [ ]:
saida = !./ex1
esperado = ["Alimento: R$ 127.00", "Limpeza: R$ 47.40", "Bebida: R$ 96.00",
            "Higiene: R$ 25.90", "Total: R$ 296.30"]
print("OK" if saida == esperado else "Ainda nao confere:\n" + "\n".join(saida))

### Exercício 2 — Data com mês nomeado

Aqui a enumeração começa em 1, porque o número tem significado externo: é o mês do calendário.

Complete `dias_no_mes` e `imprime_data` para obter:

```
7 de marco de 2005
29 de fevereiro de 2024 -- data valida
30 de fevereiro de 2025 -- data invalida
```

Considere ano bissexto quando divisível por 4 e não por 100, ou divisível por 400.

In [ ]:
%%writefile ex2.c
#include <stdio.h>

typedef enum {
    JAN = 1, FEV, MAR, ABR, MAI, JUN,
    JUL, AGO, SET, OUT, NOV, DEZ
} Mes;

typedef struct {
    int dia;
    Mes mes;
    int ano;
} Data;

int bissexto(int ano) {
    return (ano % 4 == 0 && ano % 100 != 0) || (ano % 400 == 0);
}

int dias_no_mes(Mes m, int ano) {
    /* TODO 1: retorne 31, 30 ou 28/29 conforme o mes (use switch) */
    return 0;
}

void imprime_mes(Mes m) {
    /* TODO 2: imprima o nome do mes em minusculas, sem acento
       (janeiro, fevereiro, marco, ...) usando switch */
}

int data_valida(Data d) {
    /* TODO 3: valide dia entre 1 e dias_no_mes, e mes entre JAN e DEZ */
    return 0;
}

void imprime_data(Data d) {
    printf("%d de ", d.dia);
    imprime_mes(d.mes);
    printf(" de %d", d.ano);
}

int main(void) {
    Data datas[3] = { {7, MAR, 2005}, {29, FEV, 2024}, {30, FEV, 2025} };

    imprime_data(datas[0]);
    printf("\n");
    for (int i = 1; i < 3; i++) {
        imprime_data(datas[i]);
        printf(" -- data %s\n", data_valida(datas[i]) ? "valida" : "invalida");
    }
    return 0;
}

In [ ]:
!gcc -Wall ex2.c -o ex2 && ./ex2

In [ ]:
saida = !./ex2
esperado = ["7 de marco de 2005",
            "29 de fevereiro de 2024 -- data valida",
            "30 de fevereiro de 2025 -- data invalida"]
print("OK" if saida == esperado else "Ainda nao confere:\n" + "\n".join(saida))

### Exercício 3 — Máquina de estados de um pedido

Um pedido percorre estados numa ordem fixa. A enumeração descreve os estados; a função `avanca`
descreve as transições legais.

Regras: `RECEBIDO → PREPARANDO → ENVIADO → ENTREGUE`. `ENTREGUE` não avança mais.
`CANCELADO` é um estado final: também não avança.

Saída esperada:

```
Pedido 1001: Recebido -> Preparando
Pedido 1002: Enviado -> Entregue
Pedido 1003: Entregue -> Entregue
Pedido 1004: Cancelado -> Cancelado
```

In [ ]:
%%writefile ex3.c
#include <stdio.h>

typedef enum {
    RECEBIDO, PREPARANDO, ENVIADO, ENTREGUE, CANCELADO, NUM_ESTADOS
} Estado;

typedef struct {
    int    numero;
    char   cliente[20];
    double valor;
    Estado estado;
} Pedido;

void imprime_estado(Estado e) {
    /* TODO 1: imprima Recebido, Preparando, Enviado, Entregue ou Cancelado */
}

Estado avanca(Estado e) {
    /* TODO 2: devolva o proximo estado segundo as regras do enunciado.
       Cuidado: nao basta escrever e + 1. */
    return e;
}

int main(void) {
    Pedido pedidos[4] = {
        {1001, "Ana",   89.90, RECEBIDO},
        {1002, "Bruno", 45.00, ENVIADO},
        {1003, "Carla", 12.50, ENTREGUE},
        {1004, "Diego", 70.00, CANCELADO}
    };

    for (int i = 0; i < 4; i++) {
        printf("Pedido %d: ", pedidos[i].numero);
        imprime_estado(pedidos[i].estado);
        printf(" -> ");
        imprime_estado(avanca(pedidos[i].estado));
        printf("\n");
    }
    return 0;
}

In [ ]:
!gcc -Wall ex3.c -o ex3 && ./ex3

In [ ]:
saida = !./ex3
esperado = ["Pedido 1001: Recebido -> Preparando",
            "Pedido 1002: Enviado -> Entregue",
            "Pedido 1003: Entregue -> Entregue",
            "Pedido 1004: Cancelado -> Cancelado"]
print("OK" if saida == esperado else "Ainda nao confere:\n" + "\n".join(saida))

**Pergunta:** por que `e + 1` seria uma implementação errada de `avanca`, mesmo produzindo o
resultado certo para `RECEBIDO`? Relacione com o que vimos sobre `enum` não ser um tipo fechado.

### Exercício 4 — O melhor de cada categoria

Reaproveite `Produto` do Exercício 1 e encontre o produto **mais caro de cada categoria**.
Guarde o índice do campeão num vetor indexado pela enumeração, inicializado com `-1` para
"categoria ainda sem produto".

Saída esperada:

```
Alimento: Arroz 5kg (R$ 27.50)
Limpeza: Sabao em po (R$ 15.00)
Bebida: Suco de uva (R$ 10.00)
Higiene: (nenhum)
```

In [ ]:
%%writefile ex4.c
#include <stdio.h>

typedef enum { ALIMENTO, LIMPEZA, BEBIDA, HIGIENE, NUM_CATEGORIAS } Categoria;

typedef struct {
    char      nome[30];
    Categoria categoria;
    double    preco;
    int       quantidade;
} Produto;

void imprime_categoria(Categoria c) {
    switch (c) {
        case ALIMENTO: printf("Alimento"); break;
        case LIMPEZA:  printf("Limpeza");  break;
        case BEBIDA:   printf("Bebida");   break;
        case HIGIENE:  printf("Higiene");  break;
        case NUM_CATEGORIAS: break;
    }
}

int main(void) {
    Produto estoque[6] = {
        {"Arroz 5kg",    ALIMENTO, 27.50, 3},
        {"Feijao 1kg",   ALIMENTO,  8.90, 5},
        {"Detergente",   LIMPEZA,   2.90, 6},
        {"Sabao em po",  LIMPEZA,  15.00, 2},
        {"Refrigerante", BEBIDA,    7.00, 8},
        {"Suco de uva",  BEBIDA,   10.00, 4}
    };
    int n = 6;
    int campeao[NUM_CATEGORIAS];

    /* TODO 1: inicialize campeao com -1 para todas as categorias */

    /* TODO 2: para cada produto, atualize o campeao da sua categoria */

    /* TODO 3: imprima uma linha por categoria; quando nao houver produto,
       imprima "<Categoria>: (nenhum)" */

    return 0;
}

In [ ]:
!gcc -Wall ex4.c -o ex4 && ./ex4

In [ ]:
saida = !./ex4
esperado = ["Alimento: Arroz 5kg (R$ 27.50)",
            "Limpeza: Sabao em po (R$ 15.00)",
            "Bebida: Suco de uva (R$ 10.00)",
            "Higiene: (nenhum)"]
print("OK" if saida == esperado else "Ainda nao confere:\n" + "\n".join(saida))

### Exercício 5 — A armadilha do `typedef` de vetor

Este exercício não tem TODO: é uma **medição**. Preveja os quatro números impressos antes de rodar.

In [ ]:
%%writefile ex5.c
#include <stdio.h>
#include <string.h>

typedef char Nome[20];

typedef struct {
    Nome nome;   /* o mesmo vetor, agora dentro de um registro */
} Registro;

void por_apelido(Nome x)     { printf("dentro de por_apelido: %zu\n", sizeof(x)); }
void por_registro(Registro r) { printf("dentro de por_registro: %zu\n", sizeof(r.nome)); }

int main(void) {
    Nome n;
    strcpy(n, "Ana Souza");
    Registro r;
    strcpy(r.nome, "Ana Souza");

    printf("no main, sizeof(Nome) = %zu\n", sizeof(Nome));
    printf("no main, sizeof(r)    = %zu\n", sizeof(r));
    por_apelido(n);
    por_registro(r);
    return 0;
}

In [ ]:
!gcc -Wall ex5.c -o ex5 && ./ex5

**Escreva a conclusão:** por que `sizeof` dá resultados diferentes dentro de `por_apelido` e
dentro de `por_registro`, se os dois recebem "o mesmo" `Nome`? Qual das duas assinaturas você
usaria numa função que precisa da cópia dos 20 bytes?

Repare também no aviso que o `gcc -Wall` emite ao ver `sizeof(x)` dentro de
`por_apelido`: o compilador conhece a armadilha e avisa.


## 5. Desafio — Relatório da turma

Escreva um programa completo que:

1. declare `Aluno turma[8]` com os campos `nome`, `matricula`, `media`, `situacao` e `turno`;
2. preencha os oito alunos no próprio código (invente as médias, cobrindo as três situações);
3. classifique cada aluno a partir da média, **sem números mágicos**;
4. imprima uma tabela alinhada com nome, matrícula, média, situação e turno;
5. imprima um histograma de asteriscos com a contagem por situação;
6. imprima a média de cada turno, percorrendo a enumeração `Turno` com um laço.

Exemplo do formato do histograma:

```
Reprovado   : ** (2)
Recuperacao : *** (3)
Aprovado    : *** (3)
```

**Critérios de qualidade** (valem tanto quanto o resultado):

- compila com `gcc -Wall -Wextra` sem nenhum aviso;
- nenhum literal 0, 1 ou 2 representando situação ou turno;
- os vetores de agregação são dimensionados pela sentinela da enumeração;
- todo `switch` sobre enumeração trata todos os valores.

In [ ]:
%%writefile desafio.c
#include <stdio.h>

typedef enum { MATUTINO, VESPERTINO, NOTURNO, NUM_TURNOS } Turno;
typedef enum { REPROVADO, RECUPERACAO, APROVADO, NUM_SITUACOES } Situacao;

typedef struct {
    char     nome[20];
    int      matricula;
    double   media;
    Situacao situacao;
    Turno    turno;
} Aluno;

/* TODO: classifica, imprime_situacao, imprime_turno */

int main(void) {
    /* TODO: turma[8], classificacao, tabela, histograma e media por turno */
    return 0;
}

In [ ]:
!gcc -Wall -Wextra desafio.c -o desafio && ./desafio

### Perguntas para entregar junto com o código

1. Se amanhã a coordenação criar a situação `TRANCADO`, quantos lugares do seu programa precisam
   mudar? Quais deles o compilador aponta sozinho?
2. Meça `sizeof(Aluno)` e compare com a soma dos campos. Onde está o *padding* — e mudar a ordem
   dos campos reduziria o tamanho?
3. Você usou `typedef` em todos os tipos ou deixou algum `struct`/`enum` explícito? Justifique a
   escolha.

## Referências

- KERNIGHAN, B. W.; RITCHIE, D. M. *C: a linguagem de programação — padrão ANSI*. Campus, 1989.
  (cap. 2 e 6)
- KING, K. N. *C Programming: A Modern Approach*. 2. ed. W. W. Norton, 2008. (cap. 16)
- BACKES, A. *Linguagem C: completa e descomplicada*. Elsevier, 2013.

Lista completa em `../referencias.bib`.